In [ ]:
from src.utils.notebook_setup import *
import src.utils.notebook_ploting as nb_plot
import src.utils.features as features
setup_pandas()

In [ ]:
df = load_dataset(custom_features=True)

In [ ]:
numerical_features, categorical_features = features.split_features(df)
features_df = df[numerical_features].copy()

In [ ]:
corr_matrix = features_df.corr(method="pearson")

In [ ]:
nb_plot.correlation_heatmap(corr_matrix, is_abs = True)

In [ ]:
# limiar para comparação. threshold => 0.9 -> alta correlação
    # talvez seja necessário buscar justificativa para esse valor
threshold = 0.9
# correlação e correlação negativa tem o mesmo impacto
corr_abs = corr_matrix.abs()

# pega apenas parte de cima da matriz, tambémn ignora a diagonal. evita cálculo repetido
#ex:
# 1 2 3 -> nan  2   3
# 4 5 6 -> nan nan  3
# 7 8 9 -> nan nan nan

# np.ones(corr_abs.shape) -> cria matriz de 1 com o shape
# np.triu( -> pega apenas triângulo superior
# k = 1 exclui diagonal
# transforma em boleano
# em resumo, uma máscara boleana para a matriz

upper = corr_abs.where(
    np.triu(np.ones(corr_abs.shape), k=1).astype(bool)
)

# upper.stack() -> empilha as colunas para transformar em linhas, já filtra nan
# .reset_index() -> transforma series em tabela
# renomear os nomes de coluna que o pandas atribui
#display(upper.stack())
corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={
        "level_0": "feature_1",
        "level_1": "feature_2",
        0: "correlation"
    })
)

high_corr_pairs = corr_pairs[
    corr_pairs["correlation"] >= threshold
].sort_values(by="correlation", ascending=False)


In [ ]:
high_corr_pairs
#michaeloc9@gmail.com - overleaf , compartilhar projeto src2dst_duration_ms

In [ ]:
from src.configs.paths import ARTIFACTS_DIR
from src.io.io_utils import save_parquet

path = ARTIFACTS_DIR / "high_correlation_pairs.parquet"
save_parquet(high_corr_pairs, path)
